# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**

* **Nombres y matrículas:**

- Gabriela del Carmen González Domínguez - A01796282
- Bertha Itzel Salamanca Murcia - A01797439
- Omar Aguilar Macedo - A01797078


* **Número de Equipo:**
20

* ##### **El formato de este cuaderno de Jupyter es libre, pero incluye al menos lo solicitado en el archivo PDF asociado a esta actividad.**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**

In [1]:
# @title Instalar Librerías
!pip install pymupdf4llm # PDFs a Markdown directamente
!pip install pypdfium2 # transforma páginas de pdf a imagen, para leer con LLM
!pip install qwen-vl-utils # para Qwen 3.2 Vision instruct
!pip install bitsandbytes # cambiar la cuantización del modelo base
!pip install langchain langchain-core langchain-chroma langchain-huggingface # Langchain + Chroma for RAG
!pip install langchain_text_splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

In [2]:
# @title Importar Librerías
import os
import gc
import re
from pathlib import Path

from pymupdf4llm import to_markdown
import pypdfium2 as pdfium
import cv2

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from qwen_vl_utils import process_vision_info

from langchain_core.documents import Document

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

## Lectura de Archivos

In [ ]:
#@title Utils

def save_and_get_markdown(file_base, to_markdown_fn, output_dir="."):
  if not os.path.exists(f"{output_dir}/{file_base}.md"):
    print(f'transforming pdf to markdown: {file_base}')
    markdown = to_markdown_fn(file_base)
    with open(f"{output_dir}/{file_base}.md", "w") as f:
      f.write(markdown)
  else:
    print(f'markdown already exists: {file_base}')
    with open(f"{output_dir}/{file_base}.md", "r") as f:
      markdown = f.read()
  return markdown


### Python Cheatsheet

El Layout de este PDF es relativamente simple para leer con la ayuda de lectores de pdf, se validaron diversas librerías como: `PyMuPDF`, `pdfplumber`, `pymupdf4llm`, `pypdf`, `marker-pdf`.

La librería que obtuvo mejores resultados fue `pymupdf4llm`, la cual también tiene la ventaja de que puede mandar el resultado directamente en formato markdown.


In [ ]:
# for the case of the python cheatsheet, the layout was simpler
# and pymupdf4llm did a decent work parsing it and returning the output
# as markdown

pc_markdown = save_and_get_markdown(
    "python_cheatsheet",
    lambda file: to_markdown(f"{file}.pdf")
)

markdown already exists: python_cheatsheet


In [ ]:
print("python cheatsheet markdown sample\n")
pc_markdown[0:500]

python cheatsheet markdown sample



'## **Real Python Pocket Reference** \n\nVisit realpython.com to turbocharge your Python learning with in-depth tutorials, real-world examples, and expert guidance. \n\n## **Getting Started** \n\nFollow these guides to kickstart your Python journey: realpython.com/what-can-i-do-with-python realpython.com/installing-python realpython.com/python-first-steps \n\n## **Start the Interactive Shell** \n\n- $ python \n\n## **Quit the Interactive Shell** \n\n>>> exit() \n\n## **Run a Script** \n\n- $ python my_script.py \n\n'

### Machine Learning Cheatsheet
El cheatsheet de Machine Learning, tuvo un reto mayor por su layout, que combina estructuras anidadas con texto vertical y horizontal asi como tablas. La mayoría de las librerias de lectura de PDF fallaron al leer el texto vertical, y los que pudieron leerlo requerían un gran esfuerzo para poder limpiarlo y obtener un texto usable.

La decisión fue usar un modelo LLM de vision de código abierto `Qwen/Qwen2.5-VL-7B-Instruct`, que a pesar de que el procesamiento es más pesado, fue capaz de leer una imagen del pdf y producir un resultado decente y en formato markdown.

El proceso para transformar este PDF consistión en varios pasos:
1. Transformar el PDF a imagen.
2. Transformar la imagen a escala de grises, escalandola a un 30% de su tamaño original.
3. Cargar el LLM de Vision, con cuantización de 4 bits, para poder ejecutarlo en Colab.
5. Pedirle al modelo que leyera la imagen y diera como resultado un archivo con el markdown de la imagen.

In [ ]:
#@title 1. Transformar pdf a imagen

ml_cheatsheet_base = 'ml_cheatsheet'

pdf_path = f'{ml_cheatsheet_base}.pdf'
pdf = pdfium.PdfDocument(pdf_path)

for idx, page in enumerate(pdf):
  bitmap = page.render(scale=200/72) # 72 is the default PDF points per inch
  pil_img = bitmap.to_pil()
  image_path = f'{ml_cheatsheet_base}_{str(idx).zfill(2)}.jpg'
  pil_img.save(image_path, 'JPEG')
  print(f"página {idx+1} convertida a imagen: {image_path}")

página 1 convertida a imagen: ml_cheatsheet_00.jpg


In [ ]:
# @title 2. Transformar imagen a escala de grises y escalar

image = cv2.imread(f'{ml_cheatsheet_base}_00.jpg')
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Option 1: Simple Thresholding
thresh_val, bw_image = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)
# Option 2: Otsu's Binarization
# thresh_val, bw_image = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

resized_image = cv2.resize(gray_image, None, fx=0.3, fy=0.3, interpolation=cv2.INTER_AREA)

# Save the result
cv2.imwrite(f'{ml_cheatsheet_base}_00_r.jpg', resized_image)

True

In [ ]:
# @title 3. Cargar LLLM

# Configuración de 4-bit BitsAndBytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Usa SDPA attention para prevenir memory leaks
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa"
)

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28   # <- Caps resolution to clean up memory
)


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
#@title 4. Pedirle al LLM el markdown de la imagen

def get_markdown_from_image(base_file):
  # Prompt
  messages = [
      {
          "role": "user",
          "content": [
              {"type": "image", "image": f"{base_file}_00_r.jpg"},
              {"type": "text", "text": "Please read and transcribe all the visible text from this document image exactly as it appears. The image has vertical and horizontal text, The vertical text are headings, the horizontal are tables. Format the output with markdown"}
          ]
      }
  ]

  # Generar inputs del mensaje, process_vision_info carga la imagen automáticamente
  text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  image_inputs, video_inputs = process_vision_info(messages)

  inputs = processor(
      text=[text],
      images=image_inputs,
      videos=video_inputs,
      padding=True,
      return_tensors="pt"
  ).to("cuda")

  generated_ids = model.generate(**inputs, max_new_tokens=2048)
  generated_ids_trimmed = [
      out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]

  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )

  return output_text[0]

In [ ]:
mlc_markdown = save_and_get_markdown("ml_cheatsheet", get_markdown_from_image)

transforming pdf to markdown: ml_cheatsheet


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
print("ml cheatsheet markdown sample\n")
mlc_markdown[0:500]

ml cheatsheet markdown sample



'```markdown\n# Top Machine Learning Algorithms\n\n## Supervised Learning\n\n### Linear Models\n\n| ALGORITHM | DESCRIPTION | APPLICATIONS | ADVANTAGES | DISADVANTAGES |\n|-----------|-------------|--------------|-------------|---------------|\n| Linear Regression | A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable | Use CASES | 1. Stock price prediction<br>2. Predicting customer lifetime value | 1. Explicable method<br>2. Interpretable by its o'

In [ ]:
#@title Clean Memory for LLM

# Borrar variables
if 'model' in globals(): del model
if 'processor' in globals(): del processor

# Forzar el GC de python y limpieza de cache de CUDA
gc.collect()
torch.cuda.empty_cache()

## RAG + LLM

### Generación de Embeddings y Base de Datos Vectorial (ChromaDB)

In [49]:
def clean_markdown_text(text: str) -> str:
    text = text.strip()

    # Quitar fence global ```markdown ... ```
    text = re.sub(r"^\s*```markdown\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)

    # Corregir el pipe extra en "USE CASES |"
    text = text.replace("Use CASES |", "Use CASES ")

    # Opcional: normalizar saltos de línea
    text = text.replace("\r\n", "\n")

    return text.strip()


In [50]:
def native_directory_loader(directory_path: str, glob_pattern: str = "*.md") -> list[Document]:
    """
    Mimics DirectoryLoader + TextLoader using native Python.
    Scans a directory for matching files and loads them as LangChain Documents.
    """
    documents = []
    base_path = Path(directory_path)

    for file_path in base_path.glob(glob_pattern):
        if file_path.is_file():
            try:
                text_content = file_path.read_text(encoding="utf-8")
                text_content = clean_markdown_text(text_content)
                doc = Document(
                    page_content=text_content,
                    metadata={"source": str(file_path)}
                )
                documents.append(doc)

            except Exception as e:
                print(f"Skipping {file_path} due to error: {e}")

    return documents

In [51]:
raw_documents = native_directory_loader('.', glob_pattern="*.md")

In [52]:
for doc in raw_documents:
  print(doc.metadata)
  print(doc.page_content[:100])
  print('\n---------\n')

{'source': 'python_cheatsheet.md'}
## **Real Python Pocket Reference** 

Visit realpython.com to turbocharge your Python learning with 

---------

{'source': 'ml_cheatsheet.md'}
# Top Machine Learning Algorithms

## Supervised Learning

### Linear Models

| ALGORITHM | DESCRIPT

---------



In [54]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

md_header_splits = []

for doc in raw_documents:
    splits = markdown_splitter.split_text(doc.page_content)

    # Reinyectar metadata original, porque el splitter genera nuevos Documents
    for split in splits:
        split.metadata.update(doc.metadata)

    md_header_splits.extend(splits)


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
    # separators=["\n\n", "\n", ". ", " ", ""]
)

final_splits = text_splitter.split_documents(md_header_splits)

# initialize embeddings and chromadb
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # Multilingual M3 version
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)


# persist the vector database locally to the './chroma_db' directory
vector_store = Chroma.from_documents(
    documents=final_splits,
    embedding=embedding_model,
    persist_directory="./chroma_db4"
)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [53]:
import shutil
from pathlib import Path

db_path = Path("./chroma_db3")

if db_path.exists():
    shutil.rmtree(db_path)


In [ ]:
# !top -b -n 1 -o %MEM | head -n 20

In [55]:
#@title Prueba de Búsqueda de DB vectorial
query = "What is logistic regression?"

docs = vector_store.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(docs, start=1):
    print("\n----------\n")
    print(f"RESULTADO {i}")
    print("Score:", score)
    print("Metadata:", doc.metadata)
    print(doc.page_content[:1000])


----------

RESULTADO 1
Score: 0.7137578725814819
Metadata: {'Header 1': 'Top Machine Learning Algorithms', 'source': 'ml_cheatsheet.md', 'Header 3': 'Linear Models', 'Header 2': 'Supervised Learning'}
### Linear Models  
| ALGORITHM | DESCRIPTION | APPLICATIONS | ADVANTAGES | DISADVANTAGES |
|-----------|-------------|--------------|-------------|---------------|
| Linear Regression | A simple algorithm that models a linear relationship between inputs and a continuous numerical output variable | Use CASES  1. Stock price prediction<br>2. Predicting customer lifetime value | 1. Explicable method<br>2. Interpretable by its output coefficients<br>3. Faster to train than other machine learning methods | 1. Assumes linearity between inputs and outputs<br>2. Can be overfit with small, high-dimensional data |  
| Logistic Regression | A simple algorithm that models a linear relationship between inputs and a categorical output (0 or 1) | Use CASES  1. Credit risk prediction<br>2. Customer ch

In [56]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

### LLM + RAG

In [14]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit config keeps memory minimal while preserving 99% logic capability
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'max_new_tokens', 'eos_token_id', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [57]:
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.05
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [15]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate.from_template("""
You are an assistant answering questions using only the provided context.

Rules:
- Answer only with information found in the context.
- If the context does not contain the answer, say: "I don't have enough information in the provided documents."
- Be concise.
- When useful, mention the source file or section.

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    formatted = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown source")
        h1 = doc.metadata.get("Header 1", "")
        h2 = doc.metadata.get("Header 2", "")
        h3 = doc.metadata.get("Header 3", "")

        header_path = " > ".join([h for h in [h1, h2, h3] if h])

        formatted.append(
            f"Source: {source}\n"
            f"Section: {header_path}\n\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted)


In [58]:
def clean_model_output(text):
    stop_markers = [
        "\nHuman:",
        "\nUser:",
        "\nAssistant:",
        "\nContext:",
        "\nQuestion:"
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    return text.strip()


def ask_qwen_with_context(question, docs):
    context = format_docs(docs)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a question-answering assistant for a RAG system. "
                "Each question is independent. "
                "Answer only using the provided context. "
                "Do not continue the conversation. "
                "Do not generate new questions. "
                "Do not write Human:, User:, or Assistant:. "
                "If the context does not contain the answer, say only: "
                "\"I don't have enough information in the provided documents.\""
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question:\n{question}\n\n"
                "Answer:"
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt",
        truncation=True,
        max_length=8192
    ).to(model.device)

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            repetition_penalty=1.08,
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return clean_model_output(answer)

In [59]:
message = "What is the difference between supervised and unsupervised learning?"
docs = retriever.invoke(message)
ask_qwen_with_context(message, docs)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


"The context provided does not contain information about the differences between supervised and unsupervised learning. Therefore, I don't have enough information in the provided documents."

In [60]:
message = "Según la sección de 'Exceptions', ¿qué excepción se lanza si divido entre cero?"
docs = retriever.invoke(message)
ask_qwen_with_context(message, docs)

'ZeroDivisionError'

In [61]:
preguntas = [
 "Según la sección de 'Exceptions', ¿qué excepción se lanza si divido entre cero?",
 '¿Puedes proporcionarme 2 ejemplos de métodos de cadena (string methods)?',
 '¿Cuáles son las principales desventajas de utilizar el modelo Random Forests?',
 '¿Cuáles son 3 casos de uso (use cases) de las técnicas de aprendizaje no supervisado (Unsupervised Learning) para las técnicas de agrupamiento (clustering)?',
 '¿En python, Cómo puedo generar un loop que vaya de 0 a 10?',
 'Menciona 2 ejemplos de algorimtos basados en modelos lineales',
  "¿Cuáles son las mejores librerías de juegos en python?",
  'Menciona el significado de la vida.'
]

for pregunta in preguntas:
  docs = retriever.invoke(pregunta)
  respuesta = ask_qwen_with_context(pregunta, docs)
  print(f"\n---------\n")
  print(f"Pregunta: {pregunta}")
  print(f"Respuesta: {respuesta}")


---------

Pregunta: Según la sección de 'Exceptions', ¿qué excepción se lanza si divido entre cero?
Respuesta: ZeroDivisionError

---------

Pregunta: ¿Puedes proporcionarme 2 ejemplos de métodos de cadena (string methods)?
Respuesta: "a".upper()                     # "A" "a".strip()                   # "a"

---------

Pregunta: ¿Cuáles son las principales desventajas de utilizar el modelo Random Forests?
Respuesta: Las principales desventajas de utilizar el modelo Random Forests según la tabla proporcionada son:
1. El entrenamiento puede ser complejo y耗时
用户提供了一个关于随机森林模型的表格，询问其主要缺点。根据提供的信息，随机森林的主要缺点包括：
1. 培训复杂性可能很高。
2. 不容易解释。

---------

Pregunta: ¿Cuáles son 3 casos de uso (use cases) de las técnicas de aprendizaje no supervisado (Unsupervised Learning) para las técnicas de agrupamiento (clustering)?
Respuesta: 1. Segmentación de clientes
2. Sistemas de recomendación
3. Detección de tipos de archivos

---------

Pregunta: ¿En python, Cómo puedo generar un loop que vaya de 0 a 10?
Re

In [62]:
preguntas = [
 'Which are the disadvantages of using a Random Forest model?',
]

for pregunta in preguntas:
  docs = retriever.invoke(pregunta)
  respuesta = ask_qwen_with_context(pregunta, docs)
  print(f"\n---------\n")
  print(f"Pregunta: {pregunta}")
  print(f"Respuesta: {respuesta}")


---------

Pregunta: Which are the disadvantages of using a Random Forest model?
Respuesta: The disadvantages of using a Random Forest model include:
1. Training complexity can go high.
2. Not easy to interpret.


### Nota sobre el comportamiento del sistema RAG

Durante las pruebas del sistema RAG se observaron dos comportamientos relevantes que vale la pena considerar en una implementación para uso cotidiano o en producción.

Primero, al realizar la pregunta **¿Cuáles son las diferencias entre aprendizaje supervisado y no supervisado?**, el sistema respondió que no contaba con suficiente información en los documentos proporcionados. Aunque el documento contiene secciones relacionadas con algoritmos supervisados y no supervisados, no incluye una definición explícita de ambos conceptos ni una comparación directa entre ellos. Debido a que el prompt fue configurado para responder únicamente con base en el contexto recuperado, esta respuesta se considera adecuada, ya que evita que el modelo utilice conocimiento externo o genere una respuesta no sustentada directamente en los documentos.

Segundo, al realizar la pregunta **¿Cuáles son las principales desventajas de utilizar el modelo Random Forests?**, el sistema sí recuperó la información correcta desde el documento, identificando como desventajas que la complejidad del entrenamiento puede ser alta y que el modelo no es fácil de interpretar. Sin embargo, cuando la pregunta fue formulada en español, el modelo generó una respuesta parcialmente mezclada con caracteres en chino. Al formular la misma pregunta en inglés, la respuesta fue más estable y coherente. Esto sugiere que, aunque el proceso de recuperación funcionó correctamente, la etapa de generación puede presentar problemas de control de idioma cuando se utilizan modelos multilingües locales y documentos en un idioma distinto al de la pregunta.

Estos casos muestran que un sistema RAG no depende únicamente de recuperar documentos relevantes. También es necesario cuidar la calidad del contenido fuente, la segmentación de los documentos, el idioma de las preguntas, las instrucciones del prompt y los mecanismos de validación de la salida. En un entorno de producción, sería recomendable incluir controles adicionales como normalización de idioma, validación de caracteres inesperados, límites de generación, revisión de fuentes recuperadas y respuestas de respaldo cuando el contexto no sea suficiente.

### Chat with Gradio

In [65]:
import gradio as gr
import time

def rag_chat(message, history):
    docs = retriever.invoke(message)

    # Debug para revisar si el retrieval está funcionando bien
    # print("=" * 80)
    # print("QUESTION:", message)

    # for i, doc in enumerate(docs, start=1):
    #     print(f"\nDOC {i}")
    #     print("Metadata:", doc.metadata)
    #     print(doc.page_content[:500])

    response = ask_qwen_with_context(message, docs)
    for i in range(len(response)):
        time.sleep(0.03)
        yield response[:i+1]

demo = gr.ChatInterface(
    fn=rag_chat,
    type="messages",
    title="RAG con PDFs en Markdown",
    description="Haz preguntas sobre tus documentos Markdown generados desde PDF.",
    examples=[
        "What is logistic regression?",
        "Mention 3 examples of Exceptions in python"
    ]
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8b986a93733833f956.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8b986a93733833f956.gradio.live


# **Conclusiones:**

* #### **Incluyan sus conclusiones de la actividad chatbot LLM + RAG para documentos con información tabular:**

Para esta actividad se utilizaron dos documentos en formato PDF como base de conocimiento para el sistema RAG: un Python Cheatsheet y un Machine Learning Cheatsheet. Ambos archivos fueron convertidos a formato Markdown antes de ser procesados, con el objetivo de conservar la mayor cantidad posible de estructura semántica, como títulos, subtítulos, listas y tablas.

Asimismo se comprobó el uso de RAG con LLMs como una estrategia para limitar la interacción de un modelo de lenguaje y hacer que sus respuestas se basen únicamente en una base de conocimiento específica. Este enfoque permite reducir alucinaciones y controlar mejor la información utilizada por el modelo, siempre que los documentos recuperados contengan el contexto suficiente para responder la pregunta del usuario.

Uno de los principales retos identificados fue la lectura, limpieza y preparación de los documentos que conforman la base de conocimiento. Debido a que la información es transformada en embeddings y almacenada en una base de datos vectorial, en este caso ChromaDB, es fundamental que el texto de entrada sea claro, coherente y esté correctamente estructurado. De lo contrario, la búsqueda semántica puede recuperar fragmentos incompletos, mal segmentados o poco relevantes.

La extracción de información desde archivos PDF también representó un reto importante. El Python Cheatsheet pudo procesarse adecuadamente con herramientas convencionales, especialmente con pymupdf4llm. Sin embargo, el Machine Learning Cheatsheet requirió una estrategia distinta debido a su diseño más complejo, con texto vertical, texto horizontal y tablas. En este caso, el uso de un modelo LLM de visión permitió obtener una salida en Markdown más útil que la generada por librerías tradicionales, aunque fue necesario aplicar una etapa adicional de limpieza.

En cuanto al funcionamiento del sistema RAG, se observó que un prompt restrictivo ayuda a que el modelo responda únicamente con información contenida en los fragmentos recuperados. Esto es positivo para reducir respuestas inventadas, pero también limita la capacidad del modelo para responder preguntas conceptuales generales cuando los documentos no contienen definiciones explícitas. Por ejemplo, aunque el documento incluye secciones sobre algoritmos supervisados y no supervisados, no proporciona una explicación directa sobre las diferencias entre ambos tipos de aprendizaje. En ese caso, el sistema respondió que no contaba con suficiente información, lo cual se considera un comportamiento esperado dentro de un enfoque RAG conservador.

Finalmente, también se observó que la generación de respuestas puede presentar problemas relacionados con el idioma, especialmente cuando se utilizan modelos multilingües locales y los documentos se encuentran en un idioma distinto al de la pregunta. Esto evidencia que, para una implementación en producción, no basta con construir una base vectorial funcional; también es necesario incorporar mecanismos de control, validación y limpieza de salida para asegurar que las respuestas sean coherentes, estén en el idioma esperado y se mantengan alineadas con la información recuperada.

# **Fin de la actividad chatbot: LLM + RAG**